# Workday org data — automatic additive enrichment or standalone landing

**When an Entra baseline exists, Entra is authoritative.**
Enrichment never updates an existing baseline field, even when it is NULL or blank,
and never adds Workday-only people. Without a baseline, standalone mode creates
an email-keyed org table from Workday. This is a variant of
`Copilot_Org_Data_Workday_Lander.ipynb`; it contains no export data or credentials.

## Recipient run order
1. If Entra data is available, run `Copilot_Org_Data_Direct_Ingester` to produce a fresh
   Graph/Entra snapshot at `dbo.copilot_org_data`. `PersonId` must contain the comparable
   email identity. If no Entra table exists, AUTO uses Workday standalone instead.
2. Upload the **full Workday CSV** (not just Persona) to `Files/org_workday/`.
   Remove obsolete exports; files in one run must have identical headers.
3. Import this `.ipynb` into Fabric and attach the intended Lakehouse.
4. Check the configuration, then run all cells. The safe default writes only
   `dbo.copilot_org_data_workday_preview`, not the baseline.
5. Review the preview's counts, match rate, added fields, schema and values.
   Check local information-governance requirements for the full export's fields.
6. Only after review, deliberately select the intended publish target. To replace
   the baseline itself, set `ALLOW_BASE_OVERWRITE = True` as well as the target.
   Re-run all cells; do not run only the write cell after changing configuration.
7. Refresh the ValueLens PBIT against the published org table. Additional attributes
   may require Power Query/model changes; this artifact is **not PBIT-tested**.

Join: trimmed, lowercase `primaryWorkEmail` → trimmed, lowercase `PersonId`.
Standalone sets `PersonId` from the configured Workday email and computes
`PersonId_Normalized` for user-level matching where the report's interaction identity
is available and agrees. It does not invent users, match missing identities, or create
Power BI model relationships. The recipient must verify their query/model bindings.
This is an **email** join, not a worker UID or Graph object GUID join. If the export
uses a different email header, explicitly set `WORKDAY_EMAIL_COLUMN`. A UID/GUID
requires an explicit upstream mapping to the same identity; do not rename it to email.

## Explicit modes
- `MODE = 'auto'` (default): check **table existence only**. An existing baseline is
  validated and enriched. Only an absent table selects standalone. Empty, malformed,
  inaccessible, or duplicate-key baselines fail; AUTO never silently falls back.
- `MODE = 'enrich'`: require a valid nonempty baseline and perform strict additive
  enrichment. An absent baseline fails.
- `MODE = 'standalone'`: deliberately use only Workday, without reading the baseline.
  Raw field values are retained; names matching canonical org fields use the canonical
  spelling (for example `COUNTRY` becomes `country`, `Job_Title` becomes `JobTitle`).
  Other new names receive only Delta sanitation. Missing canonical business attributes may be mapped from
  available Workday aliases; existing raw attributes always win. Missing Graph identity/
  hierarchy fields are typed string NULLs (no invented hierarchy); `id` defaults to the
  email when absent. `PersonId`, normalized identity, count and source metadata are added.
  Raw system metadata is rejected; a raw `PersonId` is accepted only when it is exactly
  named `PersonId` and explicitly selected as the email key.

The Workday match-rate guard applies only to enrichment. Standalone cannot validate
matches against absent Entra/interaction data. All modes reject missing/blank email keys,
conflicting duplicates and empty output. `INCLUDE_UNMATCHED_WORKDAY` must remain False:
it is never an option to expand an existing Entra population.
AUTO does not infer table provenance. If a standalone output is published to `BASE_TABLE`,
the next AUTO run sees an existing baseline and therefore preserves its existing columns.
For recurring standalone refreshes, select standalone explicitly or keep baseline and output
names separate. Never infer an Entra lineage merely from a table's name.

## Strict additive contract and reruns (enrich mode)
- Baseline columns stay first, in their original order, spelling, types and values.
  Existing metadata (`PersonId_Normalized`, `TotalEmployees`, `OrgData_Source`) is
  also untouched, including blank or stale values.
- All genuinely absent raw CSV fields are appended in source order, including Persona,
  extra fields, and source `Function`, `Location` or `Organization`. No mapped or derived
  business attributes are synthesized in enrich mode, so source values cannot be overwritten by
  Job Family/Region projections. Raw Workday values are not trimmed or blank-filled.
- Only **new** names are sanitized for Delta (spaces and `,;{}()\r\n\t=` become `_`).
  Comparison is case-insensitive and also ignores non-alphanumeric characters, covering
  both raw and sanitized names. Thus baseline `Job Title` wins over Workday `Job_Title`.
  Ambiguous normalized Workday headers fail; no arbitrary winner is selected.
- When absent, the notebook adds string `PersonId_Normalized`, string `TotalEmployees`,
  and string `OrgData_Source` (`entra+workday:additive`). These are run metadata,
  **not lineage**. Workday cannot supply missing system metadata or private helper fields.
- An already enriched baseline's Workday columns are now existing baseline columns:
  rerunning **skips them, preserving their old values**, even if the export changed.
  For updates each cycle, either ingest fresh Graph first, or keep a separate stable,
  refreshed Graph baseline and always write enrichment to a different output.
- Missing exports perform no write (an existing output stays intact). Empty files/rows,
  missing or blank keys, conflicting duplicates, Graph key fanout, below-threshold
  matches, and empty output fail before writing. Exact duplicate Workday rows collapse.
  Workday-only people are counted but excluded. At least 50% of distinct validated
  Workday identities must match by default.
- Files are read as UTF-8 CSV with multiline/quoted fields; columns remain strings.
  Headers must match across files (apart from BOM/outer header padding).
  Do not update staged files or the baseline during a run.
- Baseline names are never sanitized. If they require Delta column mapping, the
  destination must already support those names; this notebook does not silently
  change table protocols or rename baseline columns.

The final write replaces the chosen output snapshot, not individual rows. A same-table
publish materializes a private staging table first. A failed publication can leave that
staging table for inspection; the source export is never modified.

In [ ]:
# Safe preview defaults. Publish only after reviewing the preview.
SOURCE_PATH = 'Files/org_workday/'
BASE_TABLE = 'dbo.copilot_org_data'
OUTPUT_TABLE = 'dbo.copilot_org_data_workday_preview'
MODE = 'auto'  # 'auto' | 'enrich' | 'standalone'; AUTO falls back only on table absence.
WORKDAY_EMAIL_COLUMN = 'primaryWorkEmail'
INCLUDE_UNMATCHED_WORKDAY = False
MIN_WORKDAY_MATCH_RATE = 0.5
ALLOW_BASE_OVERWRITE = False

## Validation and additive transformation
All validation happens before the output write. Diagnostics contain counts, not person records.

In [ ]:
import math
import re
import uuid
import notebookutils
from pyspark.sql import functions as F

_INVALID = re.compile(r'[ ,;{}()\r\n\t=]')
_BASE_KEY = '_vlwd_base_key'
_WORKDAY_KEY = '_vlwd_workday_key'
_METADATA = ('PersonId_Normalized', 'TotalEmployees', 'OrgData_Source')


def _norm(name):
    return re.sub(r'[^a-z0-9]', '', name.casefold())


def _safe_name(name):
    return _INVALID.sub('_', name)


def _col(name):
    return F.col('`' + name.replace('`', '``') + '`')


def _qualified(alias, name):
    return F.col(alias + '.`' + name.replace('`', '``') + '`')


def _email(column):
    return F.lower(F.trim(column.cast('string')))


def _validate_config():
    if MODE not in ('auto', 'enrich', 'standalone'):
        raise ValueError("MODE must be 'auto', 'enrich' or 'standalone'.")
    if INCLUDE_UNMATCHED_WORKDAY is not False:
        raise ValueError('INCLUDE_UNMATCHED_WORKDAY must be False: Entra population is authoritative.')
    if (isinstance(MIN_WORKDAY_MATCH_RATE, bool)
            or not isinstance(MIN_WORKDAY_MATCH_RATE, (int, float))
            or not math.isfinite(MIN_WORKDAY_MATCH_RATE)
            or not 0 <= MIN_WORKDAY_MATCH_RATE <= 1):
        raise ValueError('MIN_WORKDAY_MATCH_RATE must be finite and between 0 and 1.')
    # Restrict identifiers so the same-table guard cannot be bypassed with quoting.
    for name in (BASE_TABLE, OUTPUT_TABLE):
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*', name):
            raise ValueError('Use an unquoted schema.table identifier for BASE_TABLE and OUTPUT_TABLE.')
    if BASE_TABLE.casefold() == OUTPUT_TABLE.casefold() and ALLOW_BASE_OVERWRITE is not True:
        raise ValueError('Publishing over BASE_TABLE requires deliberate ALLOW_BASE_OVERWRITE = True.')
    if not isinstance(WORKDAY_EMAIL_COLUMN, str) or not _norm(WORKDAY_EMAIL_COLUMN):
        raise ValueError('WORKDAY_EMAIL_COLUMN must explicitly identify an email header.')


def _validate_names(names, workday=False):
    seen = set()
    for name in names:
        key = _norm(name)
        if not key:
            raise ValueError('Empty or unusable column name.')
        if key.startswith('vlwd') or key in ('wdjoin', 'basejoin'):
            raise ValueError('Reserved private helper column in input schema.')
        identity = key if workday else name.casefold()
        if identity in seen:
            raise ValueError('Ambiguous headers after normalization.' if workday
                             else 'Ambiguous case-insensitive baseline columns.')
        seen.add(identity)


def enrich_additive(base, wd):
    """Return (validated additive DataFrame, aggregate diagnostics); never write."""
    _validate_config()
    _validate_names(base.columns)
    _validate_names(wd.columns, workday=True)
    person_columns = [name for name in base.columns if name.casefold() == 'personid']
    if len(person_columns) != 1:
        raise ValueError('Baseline must have a PersonId column.')
    person_column = person_columns[0]
    email_columns = [name for name in wd.columns if _norm(name) == _norm(WORKDAY_EMAIL_COLUMN)]
    if len(email_columns) != 1:
        raise ValueError('Workday must have the configured work-email column.')

    baseline_keys = {_norm(name) for name in base.columns}
    metadata_keys = {_norm(name) for name in _METADATA}
    additions, skipped = [], []
    for source in wd.columns:
        target = _safe_name(source)
        if _norm(source) in baseline_keys or _norm(target) in baseline_keys:
            skipped.append(source)
            continue
        if _norm(target) in metadata_keys:
            raise ValueError('Workday cannot provide absent system metadata.')
        additions.append((source, target))

    b = base.withColumn(_BASE_KEY, _email(_col(person_column)))
    if b.filter(_col(_BASE_KEY).isNull() | (_col(_BASE_KEY) == '')).limit(1).count():
        raise ValueError('Baseline contains blank PersonId keys.')
    if b.groupBy(_col(_BASE_KEY)).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Duplicate normalized baseline PersonId keys would cause Graph fanout.')
    base_rows = b.count()
    if base_rows == 0:
        raise ValueError('Baseline is empty; refusing empty output.')

    w = wd.withColumn(_WORKDAY_KEY, _email(_col(email_columns[0])))
    if w.filter(_col(_WORKDAY_KEY).isNull() | (_col(_WORKDAY_KEY) == '')).limit(1).count():
        raise ValueError('Workday contains blank work-email keys.')
    raw_rows = w.count()
    if raw_rows == 0:
        raise ValueError('Workday export parsed 0 rows.')
    w = w.distinct()
    if w.groupBy(_col(_WORKDAY_KEY)).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Conflicting duplicate Workday email keys; resolve the export upstream.')
    workday_rows = w.count()
    matched = b.join(w.select(_col(_WORKDAY_KEY)),
                     _col(_BASE_KEY) == _col(_WORKDAY_KEY), 'inner').count()
    match_rate = matched / workday_rows
    if match_rate < MIN_WORKDAY_MATCH_RATE:
        raise ValueError(f'Workday match rate {match_rate:.1%} is below MIN_WORKDAY_MATCH_RATE.')

    # Project baseline fields explicitly: no coalesce, drop, rename or cast of baseline values.
    side = w.select(_col(_WORKDAY_KEY), *[_col(src).alias(dst) for src, dst in additions])
    joined = b.alias('b').join(side.alias('w'),
                              _qualified('b', _BASE_KEY) == _qualified('w', _WORKDAY_KEY), 'left')
    result = joined.select(
        *[_qualified('b', name) for name in base.columns],
        *[_qualified('w', target) for _, target in additions],
    )
    if _norm('PersonId_Normalized') not in baseline_keys:
        result = result.withColumn('PersonId_Normalized', _email(_col(person_column)))
    if _norm('TotalEmployees') not in baseline_keys:
        result = result.withColumn('TotalEmployees', F.lit(str(base_rows)))
    if _norm('OrgData_Source') not in baseline_keys:
        result = result.withColumn('OrgData_Source', F.lit('entra+workday:additive'))

    # Verify using the private computed key, never existing (possibly stale) metadata.
    check = result.select(_email(_col(person_column)).alias(_BASE_KEY))
    if check.groupBy(_col(_BASE_KEY)).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Output contains duplicate normalized PersonId keys.')
    total = result.count()
    if total == 0 or total != base_rows:
        raise ValueError('Output must preserve the complete nonempty baseline population.')
    if result.schema.fields[:len(base.schema.fields)] != base.schema.fields:
        raise ValueError('Baseline schema changed unexpectedly.')
    return result, {
        'base_rows': base_rows, 'workday_rows': workday_rows,
        'exact_duplicates_collapsed': raw_rows - workday_rows,
        'matched': matched, 'match_rate': match_rate,
        'workday_only_excluded': workday_rows - matched,
        'output_rows': total, 'added_workday_columns': len(additions),
        'skipped_existing_columns': len(skipped),
    }


def land_standalone(wd):
    """Create an email-keyed Workday org snapshot, without assuming Entra exists."""
    _validate_config()
    _validate_names(wd.columns, workday=True)
    email_columns = [name for name in wd.columns if _norm(name) == _norm(WORKDAY_EMAIL_COLUMN)]
    if len(email_columns) != 1:
        raise ValueError('Workday must have the configured work-email column.')
    email_column = email_columns[0]
    for name in wd.columns:
        if _norm(name) in {_norm(item) for item in _METADATA}:
            raise ValueError('Workday cannot provide system metadata in standalone mode.')
        if _norm(name) == 'personid' and (name != 'PersonId' or email_column != name):
            raise ValueError('Raw PersonId must be exactly named PersonId and explicitly selected as the email key.')

    keyed = wd.withColumn(_WORKDAY_KEY, _email(_col(email_column)))
    if keyed.filter(_col(_WORKDAY_KEY).isNull() | (_col(_WORKDAY_KEY) == '')).limit(1).count():
        raise ValueError('Workday contains blank work-email keys.')
    raw_rows = keyed.count()
    if raw_rows == 0:
        raise ValueError('Workday export parsed 0 rows.')
    keyed = keyed.distinct()
    if keyed.groupBy(_col(_WORKDAY_KEY)).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Conflicting duplicate Workday email keys; resolve the export upstream.')
    total = keyed.count()
    graph_fields = ['displayName', 'companyName', 'managerUPN', 'accountEnabled',
                    'OrgLevel', 'HierarchyPath', 'TopOfChain_Name', 'IsManager', 'DirectReports']
    graph_fields += [f'Level{i}_Name' for i in range(15)]
    canonical_fields = graph_fields + [
        'PersonId', 'id', 'Job_Profile', 'Job_Family', 'Job_Family_Group', 'Persona',
        'Compensation_Grade', 'Worker_Type', 'Worker_SubType', 'On_Leave', 'IsOnLeave',
        'country', 'sub_Country', 'Organization', 'JobTitle', 'Function',
        'officeLocation', 'Location', 'city',
    ]
    canonical_lookup = {_norm(name): name for name in canonical_fields}
    source_targets = {name: canonical_lookup.get(_norm(name), _safe_name(name)) for name in wd.columns}
    result = keyed.select(*[_col(name).alias(source_targets[name]) for name in wd.columns])
    output_lookup = {_norm(name): name for name in result.columns}

    def append_if_absent(name, expression):
        nonlocal result
        if _norm(name) not in output_lookup:
            result = result.withColumn(name, expression)
            output_lookup[_norm(name)] = name

    def mapped_value(candidates):
        for candidate in candidates:
            actual = output_lookup.get(_norm(candidate))
            if actual is not None:
                return _col(actual)
        return F.lit(None).cast('string')

    result = result.withColumn('PersonId', _email(_col(source_targets[email_column])))
    output_lookup[_norm('PersonId')] = 'PersonId'
    append_if_absent('id', _col('PersonId'))
    # Add mappings only when absent; never overwrite real Function/Location/Organization.
    aliases = {
        'Job_Profile': ['Job Profile', 'Job Title', 'JobTitle', 'Business Title', 'Position'],
        'Job_Family': ['Job Family'],
        'Job_Family_Group': ['Job Family Group'],
        'Persona': ['Worker Persona'],
        'Compensation_Grade': ['Compensation Grade', 'Grade', 'Pay Grade'],
        'Worker_Type': ['Worker Type', 'Employee Type'],
        'Worker_SubType': ['Worker Sub Type', 'Worker Subtype', 'Employee Sub Type'],
        'On_Leave': ['On Leave', 'Leave of Absence', 'OnLeave'],
        'country': ['Location Country', 'Country/Region'],
        'sub_Country': ['Sub Country', 'Sub-Country', 'Region', 'State', 'Province'],
        'Organization': ['Job_Family_Group'],
        'JobTitle': ['Job_Profile'],
        'Function': ['Job_Family_Group'],
        'officeLocation': ['sub_Country'],
        'Location': ['sub_Country'],
        'city': ['sub_Country'],
    }
    for target, candidates in aliases.items():
        append_if_absent(target, mapped_value(candidates))
    leave = mapped_value(['On_Leave'])
    append_if_absent('IsOnLeave',
                     F.when(leave.isNull() | (F.trim(leave) == ''), F.lit(None).cast('string'))
                      .when(F.lower(F.trim(leave)).isin('1', 'true', 'yes', 'y'), F.lit('TRUE'))
                      .otherwise(F.lit('FALSE')))
    for name in graph_fields:
        append_if_absent(name, F.lit(None).cast('string'))
    result = (result.withColumn('PersonId_Normalized', _email(_col('PersonId')))
              .withColumn('TotalEmployees', F.lit(str(total)))
              .withColumn('OrgData_Source', F.lit('workday:standalone')))
    check = result.select(_email(_col('PersonId')).alias(_BASE_KEY))
    if check.groupBy(_col(_BASE_KEY)).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Output contains duplicate normalized PersonId keys.')
    if result.count() != total or total == 0:
        raise ValueError('Standalone output must contain all validated, distinct Workday workers.')
    return result, {
        'mode': 'standalone', 'workday_rows': total, 'output_rows': total,
        'exact_duplicates_collapsed': raw_rows - total, 'match_rate': None,
    }


def prepare_org_snapshot(wd):
    """AUTO is a table-existence decision, never an exception-driven fallback."""
    _validate_config()
    if MODE == 'standalone':
        return land_standalone(wd)
    if spark.catalog.tableExists(BASE_TABLE):
        result, diagnostics = enrich_additive(spark.table(BASE_TABLE), wd)
        diagnostics['mode'] = 'enrich'
        return result, diagnostics
    if MODE == 'enrich':
        raise ValueError('Baseline table is missing. Run the Graph org-data ingester first or select AUTO/standalone.')
    return land_standalone(wd)


_validate_config()

## Read the staged full CSV export
Every file's parsed header is checked before combining files. No source means no write.

In [ ]:
def _safe_ls(path):
    try:
        return list(notebookutils.fs.ls(path))
    except Exception as exc:
        if any(text in str(exc).lower() for text in ('not found', 'no such file', 'does not exist')):
            return []
        raise


def _resolve_source(path):
    if path.endswith('/'):
        return sorted(item.path for item in _safe_ls(path)
                      if not item.isDir and item.name.lower().endswith(('.csv', '.txt')))
    folder, name = path.rsplit('/', 1)
    return [path] if any(item.name == name and not item.isDir for item in _safe_ls(folder)) else []


def _csv_reader(header):
    return (spark.read.option('header', header).option('multiLine', True)
            .option('escape', '"').option('encoding', 'UTF-8')
            .option('mode', 'FAILFAST').option('inferSchema', False))


def read_workday_csv(paths):
    expected, frames = None, []
    for path in paths:
        # Read the first parsed CSV record as data so Spark cannot hide duplicate headers.
        first = _csv_reader(False).csv(path).take(1)
        if not first:
            raise ValueError('An export file is empty.')
        headers = [(value or '').replace('\ufeff', '').strip() for value in first[0]]
        _validate_names(headers, workday=True)
        if expected is not None and headers != expected:
            raise ValueError('CSV files have different headers or column order; stage one consistent export.')
        expected = headers
        frame = _csv_reader(True).csv(path)
        if len(frame.columns) != len(headers):
            raise ValueError('CSV header width mismatch.')
        frame = frame.toDF(*headers)
        if frame.limit(1).count() == 0:
            raise ValueError('An export file contains a header but no workers.')
        frames.append(frame)
    if not frames:
        raise ValueError('No CSV paths supplied.')
    combined = frames[0]
    for frame in frames[1:]:
        combined = combined.unionByName(frame)
    return combined


sources = _resolve_source(SOURCE_PATH)
result = None
if not sources:
    print('No Workday export found. No write performed; any existing output is unchanged.')
else:
    wd = read_workday_csv(sources)
    result, diagnostics = prepare_org_snapshot(wd)
    print(diagnostics)

## Write the validated snapshot
The preview is the default destination. Enrichment preserves every baseline value;
standalone creates an org snapshot from Workday without requiring Entra.
For a same-table publish, staging prevents reading and overwriting the same Delta relation.
No table is written when no export is staged. Changing configuration requires running all cells again.

In [ ]:
if sources and result is not None:
    _validate_config()
    if OUTPUT_TABLE.casefold() == BASE_TABLE.casefold():
        staging = OUTPUT_TABLE + '_workday_stage_' + uuid.uuid4().hex
        (result.write.format('delta').mode('errorifexists')
               .saveAsTable(staging))
        (spark.table(staging).write.format('delta').mode('overwrite')
              .option('overwriteSchema', 'true').saveAsTable(OUTPUT_TABLE))
        spark.sql(f'DROP TABLE {staging}')
    else:
        (result.write.format('delta').mode('overwrite')
               .option('overwriteSchema', 'true').saveAsTable(OUTPUT_TABLE))
    print(f'Rows written to {OUTPUT_TABLE}: {spark.table(OUTPUT_TABLE).count():,}')